# Whale Watch Data — Load and Combine
Loads all monthly NOAA WhaleWatch CSV files, combines them into one dataframe,
and identifies missing months in the 2020–2025 record.

In [9]:
import pandas as pd
from pathlib import Path

WHALE_DIR  = Path("data/WhaleWatch")
OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [10]:
# --- LOAD ALL FILES ---
all_files = sorted(WHALE_DIR.glob("WhaleWatchPredictions_*.csv"))
print(f"Found {len(all_files)} files")

frames = []
loaded = []

for f in all_files:
    # Parse year and month from filename e.g. WhaleWatchPredictions_2020_02.csv
    parts = f.stem.split("_")
    year  = int(parts[1])
    month = int(parts[2])

    df = pd.read_csv(f)
    df["year"]  = year
    df["month"] = month
    frames.append(df)
    loaded.append((year, month))

df_all = pd.concat(frames, ignore_index=True)
print(f"Combined shape: {df_all.shape}")
print(f"\nColumns: {list(df_all.columns)}")

Found 64 files
Combined shape: (399168, 18)

Columns: ['Unnamed: 0', 'longitude', 'latitude', 'bathy', 'bathyrms', 'sst', 'chl', 'ssh', 'sshrms', 'month', 'year', 'fitmean', 'sdfit', 'percent', 'density', 'sddens', 'upper', 'lower']


In [11]:
# --- CONVERT LONGITUDE from 0-360 to -180 to 180 ---
df_all["longitude"] = df_all["longitude"].apply(lambda x: x - 360 if x > 180 else x)
print(f"Longitude range: {df_all['longitude'].min():.2f} to {df_all['longitude'].max():.2f}")
print(f"Latitude range:  {df_all['latitude'].min():.2f} to {df_all['latitude'].max():.2f}")

Longitude range: -135.00 to -115.00
Latitude range:  30.00 to 49.00


In [12]:
# --- IDENTIFY MISSING MONTHS ---
# Build full expected set of months 2020-01 through 2025-12
expected = set()
for y in range(2020, 2026):
    for m in range(1, 13):
        expected.add((y, m))

loaded_set = set(loaded)
missing    = sorted(expected - loaded_set)

print(f"Expected months: {len(expected)}")
print(f"Loaded months:   {len(loaded_set)}")
print(f"Missing months:  {len(missing)}")
print()

if missing:
    print("Missing month list:")
    for y, m in missing:
        print(f"  {y}-{m:02d}")

Expected months: 72
Loaded months:   64
Missing months:  8

Missing month list:
  2020-08
  2022-12
  2024-04
  2024-10
  2024-11
  2025-03
  2025-09
  2025-10


In [13]:
# --- SAVE MISSING MONTHS LOG ---
log_path = OUTPUT_DIR / "whale_watch_missing_months.txt"
with open(log_path, "w") as f:
    f.write("NOAA WhaleWatch — Missing Months Log\n")
    f.write("Expected range: 2020-01 through 2025-12\n")
    f.write(f"Expected: {len(expected)} months\n")
    f.write(f"Loaded:   {len(loaded_set)} months\n")
    f.write(f"Missing:  {len(missing)} months\n\n")
    if missing:
        for y, m in missing:
            f.write(f"  {y}-{m:02d}\n")
    else:
        f.write("  None — all months present\n")

print(f"Missing months log saved to {log_path}")

Missing months log saved to data\processed\whale_watch_missing_months.txt


In [14]:
# --- CHECK NA VALUES ---
print("NA counts per column:")
print(df_all.isnull().sum())
print()

# Rows where model output is NA (fitmean is null)
na_rows = df_all[df_all["fitmean"].isnull()]
print(f"Rows with no model output (fitmean = NA): {len(na_rows):,}")
print(f"Rows with model output:                   {len(df_all) - len(na_rows):,}")

NA counts per column:
Unnamed: 0         0
longitude          0
latitude           0
bathy         139264
bathyrms      139264
sst           131628
chl           136405
ssh           137057
sshrms        139623
month              0
year               0
fitmean       158383
sdfit         158383
percent       158383
density       158383
sddens        158383
upper         158383
lower         158383
dtype: int64

Rows with no model output (fitmean = NA): 158,383
Rows with model output:                   240,785


In [15]:
# --- DROP NA MODEL OUTPUT ROWS ---
# NA fitmean means the model could not produce an estimate for that cell-month
# These are excluded from analysis
df_all["has_model_output"] = df_all["fitmean"].notna()

df_clean = df_all.copy()
print(f"Total rows: {len(df_clean):,}")
print(f"Rows with model output:    {df_clean['has_model_output'].sum():,}")
print(f"Rows without model output: {(~df_clean['has_model_output']).sum():,}")

Total rows: 399,168
Rows with model output:    240,785
Rows without model output: 158,383


In [16]:
# --- QUICK SUMMARY ---
print("Years present:", sorted(df_clean["year"].unique()))
print("Months per year:")
print(df_clean.groupby("year")["month"].nunique())
print()
print("Whale density (fitmean) summary:")
print(df_clean["fitmean"].describe())
print()
print("Presence probability (percent) summary:")
print(df_clean["percent"].describe())

Years present: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Months per year:
year
2020    11
2021    12
2022    11
2023    12
2024     9
2025     9
Name: month, dtype: int64

Whale density (fitmean) summary:
count    240785.000000
mean          0.156980
std           0.216703
min           0.000003
25%           0.011277
50%           0.057873
75%           0.213936
max           0.996992
Name: fitmean, dtype: float64

Presence probability (percent) summary:
count    240785.000000
mean         15.698022
std          21.670324
min           0.000300
25%           1.127669
50%           5.787255
75%          21.393599
max          99.699216
Name: percent, dtype: float64


In [17]:
# --- SAVE COMBINED FILE ---
out_path = OUTPUT_DIR / "whale_watch_2020_2025_combined.csv"
df_clean.to_csv(out_path, index=False)
print(f"Saved to {out_path}")
print(f"File size: {out_path.stat().st_size / 1024 / 1024:.1f} MB")
df_clean.head()

Saved to data\processed\whale_watch_2020_2025_combined.csv
File size: 64.3 MB


,Unnamed: 0,longitude,latitude,bathy,bathyrms,sst,chl,ssh,sshrms,month,year,fitmean,sdfit,percent,density,sddens,upper,lower,has_model_output
0,1,-134.75,30.00,-4846.00,138.031570,17.737705,0.109308,0.041150,NaN,1,2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1,2,-134.75,30.25,-4825.75,189.851151,17.554877,0.111902,0.046869,NaN,1,2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2,3,-134.75,30.50,-4808.50,75.292976,17.444814,0.114568,0.040419,NaN,1,2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
3,4,-134.75,30.75,-4745.00,81.985107,17.348983,0.113918,0.034044,0.013109,1,2020,0.056487,0.022872,5.648667,0.012769,0.003799,7.935913,3.361420,True
4,5,-134.75,31.00,-4695.00,160.621368,17.108766,0.116488,0.043225,0.014208,1,2020,0.049056,0.019097,4.905648,0.011104,0.003143,6.815389,2.995906,True
